In [1]:
!pip install -q ultralytics

import sys, torch
from ultralytics import YOLO

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.8 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
import os
from pathlib import Path

SRC = Path("/kaggle/input/datasets/lazarvelinov46/anti-uav-v4-track3-yolo")
DST = Path("/kaggle/working/anti_uav_v4")

# Diagnose actual structure (Kaggle auto-extracted the zip)
print(f"Top-level contents of {SRC}:")
for p in sorted(SRC.iterdir()):
    marker = "DIR " if p.is_dir() else "FILE"
    print(f"  {marker}  {p.name}")
print()

# Find the actual data root
candidates = [SRC / "anti_uav_v4", SRC]
data_root = None
for cand in candidates:
    if (cand / "images" / "train").is_dir() and (cand / "labels" / "train").is_dir():
        data_root = cand
        break

if data_root is None:
    raise RuntimeError(f"Could not find images/train under {SRC}")

print(f"Found dataset root: {data_root}\n")

# Create a symlink — instant, no data movement
os.symlink(data_root, DST, target_is_directory=True)
print(f"Symlinked {DST} -> {data_root}\n")

# Sanity check (directory listings only — small data)
for split in ("train", "val"):
    n_img = len(list((DST / "images" / split).glob("*.jpg")))
    n_lbl = len(list((DST / "labels" / split).glob("*.txt")))
    print(f"  {split:<5}: {n_img:>7} images, {n_lbl:>7} labels  (matched: {n_img == n_lbl})")

Top-level contents of /kaggle/input/datasets/lazarvelinov46/anti-uav-v4-track3-yolo:
  DIR   anti_uav_v4

Found dataset root: /kaggle/input/datasets/lazarvelinov46/anti-uav-v4-track3-yolo/anti_uav_v4

Symlinked /kaggle/working/anti_uav_v4 -> /kaggle/input/datasets/lazarvelinov46/anti-uav-v4-track3-yolo/anti_uav_v4

  train:  122488 images,  122488 labels  (matched: True)
  val  :   30093 images,   30093 labels  (matched: True)


In [3]:
import yaml

DATA_YAML = {
    "path":  "/kaggle/working/anti_uav_v4",
    "train": "images/train",
    "val":   "images/val",
    "nc":    1,
    "names": {0: "uav"},
}

yaml_path = "/kaggle/working/data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(DATA_YAML, f, sort_keys=False, default_flow_style=False)

print(f"Wrote {yaml_path}:\n")
print(open(yaml_path).read())

Wrote /kaggle/working/data.yaml:

path: /kaggle/working/anti_uav_v4
train: images/train
val: images/val
nc: 1
names:
  0: uav



In [4]:
import shutil
from pathlib import Path

# When session 2+ runs, the checkpoint dataset (created at end of session 1)
# will be mounted under this path — if you've added it via "Add Data" in
# the notebook sidebar. Session 1 won't have this, so we just start fresh.
CKPT_DATASET = Path("/kaggle/input/uav-baseline-checkpoints")
RUN_DIR = Path("/kaggle/working/runs/baseline")
WEIGHTS_DIR = RUN_DIR / "weights"
TARGET = WEIGHTS_DIR / "last.pt"

if TARGET.exists():
    print(f"last.pt already present at {TARGET} — nothing to restore.")
elif CKPT_DATASET.exists() and (CKPT_DATASET / "last.pt").exists():
    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(CKPT_DATASET / "last.pt", TARGET)
    print(f"Restored last.pt from previous session ({CKPT_DATASET / 'last.pt'}).")
    
    # Carry over training history so the results plot stays continuous
    src_results = CKPT_DATASET / "results.csv"
    if src_results.exists():
        shutil.copy(src_results, RUN_DIR / "results.csv")
        print(f"Also restored results.csv ({src_results.stat().st_size} bytes).")
else:
    print("No previous checkpoint found → training will start fresh from yolov8s.pt.")

No previous checkpoint found → training will start fresh from yolov8s.pt.


In [5]:
from pathlib import Path
from ultralytics import YOLO

ckpt = Path("/kaggle/working/runs/baseline/weights/last.pt")

if ckpt.exists():
    print(f"Resuming from {ckpt}")
    model = YOLO(str(ckpt))
    results = model.train(resume=True)
else:
    print("Starting fresh training from yolov8s.pt")
    model = YOLO("yolov8s.pt")
    results = model.train(
        data="/kaggle/working/data.yaml",
        epochs=50,
        imgsz=640,
        batch=16,
        workers=2,
        device=0,
        cache=False,
        time=9,                   # hard stop after 9 hours so the upload cell can run

        # Optimizer pinned (smoke test used 'auto' which selected AdamW)
        optimizer="SGD",
        lr0=0.01,
        momentum=0.937,
        weight_decay=0.0005,

        # Budget and checkpointing
        patience=15,
        save_period=10,           # also keep epoch10/20/30/40/50.pt for analysis

        # Reproducibility
        seed=42,
        deterministic=True,

        # Output
        project="/kaggle/working/runs",
        name="baseline",
        exist_ok=True,
        verbose=True,
    )

print("\nTraining call returned.")

Starting fresh training from yolov8s.pt
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD

In [6]:
import json, os, shutil, subprocess
from pathlib import Path

# Auth: pull KAGGLE_USERNAME / KAGGLE_KEY from notebook secrets.
# (See "One-time setup" note below — you add these once via Add-ons → Secrets.)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = secrets.get_secret("KAGGLE_KEY")

USER = "lazarvelinov46"
DATASET_SLUG = "uav-baseline-checkpoints"
DATASET_ID = f"{USER}/{DATASET_SLUG}"

RUN_DIR = Path("/kaggle/working/runs/baseline")
WEIGHTS_DIR = RUN_DIR / "weights"
STAGING = Path("/kaggle/working/ckpt_staging")

# Fresh staging dir
if STAGING.exists():
    shutil.rmtree(STAGING)
STAGING.mkdir(parents=True)

# Stage the artifacts we want persisted between sessions
for src in [WEIGHTS_DIR / "last.pt", WEIGHTS_DIR / "best.pt",
            RUN_DIR / "args.yaml", RUN_DIR / "results.csv"]:
    if src.exists():
        shutil.copy(src, STAGING / src.name)
        print(f"Staged {src.name}  ({src.stat().st_size / 1e6:.2f} MB)")
    else:
        print(f"Skip {src.name} (not present)")

# Kaggle requires a metadata file alongside the upload
metadata = {
    "title": "UAV Baseline Checkpoints",
    "id":    DATASET_ID,
    "licenses": [{"name": "CC0-1.0"}],
}
with open(STAGING / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Read the current epoch from results.csv for a meaningful version message
msg = "checkpoint upload"
results_csv = STAGING / "results.csv"
if results_csv.exists():
    lines = results_csv.read_text().strip().splitlines()
    if len(lines) > 1:
        last_epoch = lines[-1].split(",")[0].strip()
        msg = f"end of session, through epoch {last_epoch}"
print(f"\nVersion message: {msg!r}")

# Try to version an existing dataset; fall back to creating a new one.
def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.returncode

print("\nAttempting to version existing dataset...")
rc = run(["kaggle", "datasets", "version", "-p", str(STAGING), "-m", msg, "--dir-mode", "zip"])

if rc != 0:
    print("\nVersion failed — assuming dataset doesn't exist yet, creating it...")
    rc = run(["kaggle", "datasets", "create", "-p", str(STAGING), "--dir-mode", "zip"])

print("\nUpload step finished.")

Staged last.pt  (22.50 MB)
Staged best.pt  (22.50 MB)
Staged args.yaml  (0.00 MB)
Staged results.csv  (0.00 MB)

Version message: 'end of session, through epoch 15'

Attempting to version existing dataset...
Starting upload for file last.pt
Upload successful: last.pt (21MB)
Starting upload for file args.yaml
Upload successful: args.yaml (2KB)
Starting upload for file best.pt
Upload successful: best.pt (21MB)
Starting upload for file results.csv
Upload successful: results.csv (2KB)
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion

STDERR: 
  0%|          | 0.00/21.5M [00:00<?, ?B/s]
 28%|██▊       | 6.02M/21.5M [00:00<00:00, 62.8MB/s]
 79%|███████▉  | 17.0M/21.5M [00:00<00:00, 83.4MB/s]
100%|██████████| 21.5M/21.5M [00:00<00:00, 29.8MB/s]

  0%|          | 0.00/1.56k [00:00<?, ?B/s]
100%|██████████| 1.56k/1.56k [00:00<00:00, 3.68kB/s]

  0%|          | 0.00/21.5M [00:00<?, ?B/s]
 23%|██▎       | 4.95M/21.5M [00:00<00:00, 51.9